In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, random_split
import wandb
import numpy as np
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
from glob import glob
from PIL import Image

# ------------------------------
# Custom Dataset for Video Frames (no OpenCV)
# ------------------------------
class VideoFrameDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []  # list of (frame_tensor, label)
        self.transform = transform
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(sorted(os.listdir(root_dir)))}

        for class_name, label in self.class_to_idx.items():
            class_dir = os.path.join(root_dir, class_name)
            video_dirs = [d for d in os.listdir(class_dir) if os.path.isdir(os.path.join(class_dir, d))]

            for video_dir in video_dirs:
                frame_dir = os.path.join(class_dir, video_dir)
                frame_paths = sorted(glob(os.path.join(frame_dir, '*.jpg')))  # assumes frames are .jpg

                for frame_path in frame_paths:
                    self.samples.append((frame_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# ------------------------------
# Init wandb
# ------------------------------
wandb.init(
    project="ablation-veo-real",
    name="ResNet50, Epochs=15, BatchSize=32, LR=0.01, WeightDecay=1e-4",
    config={
        "epochs": 15,
        "batch_size": 32,
        "lr": 0.01,
        "optimizer": "Adam",
        "architecture": "resnet50",
        "img_size": 224,
        "split": [0.7, 0.15, 0.15],
        "weight_decay": 1e-4
    }
)
config = wandb.config

device = torch.device('cuda:3' if torch.cuda.is_available() else 'cpu')

# ------------------------------
# Transform and Dataset
# ------------------------------
transform = transforms.Compose([
    transforms.Resize((config.img_size, config.img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

dataset = VideoFrameDataset(root_dir='AI_Video_Generation_Dataset/real-veo', transform=transform)
print("Class mapping:", dataset.class_to_idx)

# ------------------------------
# Split Dataset
# ------------------------------
total_size = len(dataset)
train_size = int(config.split[0] * total_size)
val_size = int(config.split[1] * total_size)
test_size = total_size - train_size - val_size

train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])
val_indices = val_ds.indices
test_indices = test_ds.indices

train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False)

# ------------------------------
# Model Setup
# ------------------------------
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)

# ------------------------------
# Training
# ------------------------------
val_misclassified_indices = []
val_misclassified_paths = []

for epoch in tqdm(range(config.epochs), desc="Epochs"):
    model.train()
    train_loss, train_correct, total = 0.0, 0, 0

    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = 100.0 * train_correct / total
    avg_train_loss = train_loss / len(train_loader)

    # --------------------
    # Validation
    # --------------------
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]", leave=False)):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100.0 * val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_accuracy": train_acc,
        "val_loss": avg_val_loss,
        "val_accuracy": val_acc
    })

    print(f"Epoch {epoch+1}: "
          f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
          f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%")

# ------------------------------
# Save Model
# ------------------------------
torch.save(model.state_dict(), "resnet_model.pth")
wandb.save("resnet_model.pth")

# ------------------------------
# Test Evaluation
# ------------------------------
def evaluate_on_test(model, test_loader):
    model.eval()
    test_correct, test_total, test_loss = 0, 0, 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Testing"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    test_acc = 100.0 * test_correct / test_total
    avg_test_loss = test_loss / len(test_loader)

    print(f"Final Test Accuracy: {test_acc:.2f}%, Test Loss: {avg_test_loss:.4f}")
    wandb.log({
        "test_accuracy": test_acc,
        "test_loss": avg_test_loss,
        "confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=all_labels,
            preds=all_preds,
            class_names=list(dataset.class_to_idx.keys())
        )
    })

evaluate_on_test(model, test_loader)


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Class mapping: {'real': 0, 'veo3': 1}


ValueError: num_samples should be a positive integer value, but got num_samples=0